In [ ]:
library(Seurat)
library(ggplot2)
library(org.Hs.eg.db)
library(GOSemSim)
library(clusterProfiler)

In [ ]:
data <- readRDS("data/processed/sc_reference_file")
data <- subset(data, subset = annotation != "Other")
table(data@meta.data$annotation)

In [ ]:
## 提取差异基因进行富集

In [ ]:
deg_results <- read.csv("results/go/go_output_file")
ex_bdnf_genes <- deg_results[deg_results$cluster == "EX_BDNF", "gene"]
ex_bdnf_genes <- unique(ex_bdnf_genes)

In [ ]:
erich.go.BP = enrichGO(gene =ex_bdnf_genes,
                       OrgDb = org.Hs.eg.db,
                       keyType = "SYMBOL",
                       ont = "BP",
                       pvalueCutoff = 0.05,
                       qvalueCutoff = 0.05)

dotplot(erich.go.BP) + ggtitle("GO Enrichment for EX_BDNF DEGs")
write.csv(as.data.frame(erich.go.BP), "GO_enrich_BP_for_EX_BDNF.csv", row.names = FALSE)

In [ ]:
## 循环输出每个兴奋性

In [ ]:
clusters <- c("EX_ADRA1A_Vat1l", "EX_BDNF", "EX_C1QL3", "EX_env_TMEM144", "EX_ERG",
              "EX_FOXP2", "EX_HTR7_GNAL", "EX_IFITM3_CLDN5", "EX_ITGA4_TPBG", "EX_NTNG1",
              "EX_Ntng2", "EX_Nxph4_CPLX3", "EX_SLC1A3_FGFR3", "EX_TSHZ2_VWC2L")

# 遍历每个兴奋性亚群
for (cluster in clusters) {
  # 提取当前亚群的基因
  genes <- deg_results[deg_results$cluster == cluster, "gene"]
  genes <- unique(genes)
  
  # 检查基因列表是否为空
  if (length(genes) == 0) {
    cat("No genes found for cluster:", cluster, "\n")
    next
  }
  
  # 进行GO分析
  go_result <- enrichGO(gene = genes,
                        OrgDb = org.Hs.eg.db,
                        keyType = "SYMBOL",
                        ont = "BP",
                        pvalueCutoff = 0.05,
                        qvalueCutoff = 0.05)
  
  # 检查GO分析结果是否为空
  if (is.null(go_result@result) || nrow(go_result@result) == 0) {
    cat("No significant GO terms found for cluster:", cluster, "\n")
    next
  }
  
  # 打印GO分析结果的结构
  cat("GO analysis result for cluster", cluster, ":\n")
  print(head(go_result@result))
  
  # 绘制点图
  tryCatch({
    p <- dotplot(go_result) + ggtitle(paste("GO Enrichment for", cluster, "DEGs"))
    ggsave(filename = paste(cluster, "_GO_enrichment_dotplot.png"), plot = p, width = 8, height = 8)
  }, error = function(e) {
    cat("Error generating dotplot for cluster", cluster, ":\n", e$message, "\n")
  })
  
  # 保存GO分析结果到CSV文件
  write.csv(as.data.frame(go_result), paste(cluster, "_GO_enrich_BP.csv"), row.names = FALSE)
}

In [ ]:
clusters <- c("EX_ADRA1A_Vat1l", "EX_BDNF", "EX_C1QL3", "EX_env_TMEM144", "EX_ERG",
              "EX_FOXP2", "EX_HTR7_GNAL", "EX_IFITM3_CLDN5", "EX_ITGA4_TPBG", "EX_NTNG1",
              "EX_Ntng2", "EX_Nxph4_CPLX3", "EX_SLC1A3_FGFR3", "EX_TSHZ2_VWC2L")

# 遍历每个兴奋性亚群
for (cluster in clusters) {
  # 提取当前亚群的基因
  genes <- deg_results[deg_results$cluster == cluster, "gene"]
  genes <- unique(genes)
  
  # 检查基因列表是否为空
  if (length(genes) == 0) {
    cat("No genes found for cluster:", cluster, "\n")
    next
  }
  
  # 进行GO分析
  go_result <- enrichGO(gene = genes,
                        OrgDb = org.Hs.eg.db,
                        keyType = "SYMBOL",
                        ont = "BP",
                        pvalueCutoff = 0.05,
                        qvalueCutoff = 0.05)
  
  # 检查GO分析结果是否为空
  if (is.null(go_result@result) || nrow(go_result@result) == 0) {
    cat("No significant GO terms found for cluster:", cluster, "\n")
    next
  }
  
  # 绘制点图并去掉标题
  tryCatch({
    p <- dotplot(go_result)  # 不添加标题
    
    # 保存为PNG和PDF格式
    ggsave(filename = paste(cluster, "_GO_enrichment_dotplot.png"), plot = p, width = 8, height = 8)
    ggsave(filename = paste(cluster, "_GO_enrichment_dotplot.pdf"), plot = p, width = 8, height = 8)
    
    # 输出循环完成的提示
    cat("Processing for cluster", cluster, "completed.\n")
  }, error = function(e) {
    cat("Error generating dotplot for cluster", cluster, ":\n", e$message, "\n")
  })
  
  # 保存GO分析结果到CSV文件
  write.csv(as.data.frame(go_result), paste(cluster, "_GO_enrich_BP.csv"), row.names = FALSE)
}